# Imports

In [39]:
from langchain_groq import ChatGroq # for LLM
from langchain_huggingface.embeddings import HuggingFaceEmbeddings # for embeddings model
from langchain.tools import tool # for custome tools
from langchain_community.document_loaders import PyPDFLoader # for loading PDF file
from langchain_text_splitters import RecursiveCharacterTextSplitter # for chunking data
from langchain_community.vectorstores import FAISS # for vector db
# youtube search tool
from langchain_community.tools import YouTubeSearchTool
from langchain_community.document_loaders import YoutubeLoader
import ast # convert string into actual python list
# wikipedia search tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
# google search tools
import os
from langchain_community.utilities import SearchApiAPIWrapper
from langchain_core.tools import Tool
# dot env loading
from pathlib import Path
from dotenv import load_dotenv

In [40]:
env_path = Path("../.env")    # go up one folder
load_dotenv(dotenv_path=env_path)

True

# model

In [17]:
llm = ChatGroq(model="qwen/qwen3-32b")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


# Tools

In [5]:
tools = []

Extract data tool.

In [43]:
@tool("extract_data")
def extract_data(pdf_path: str) -> str:
    """Extracts raw text from a PDF file."""
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    return "\n\n".join([doc.page_content for doc in docs])

tools.append(extract_data)

Chunk data tool

In [44]:
@tool("chunk_data")
def chunk_data(raw_text: str):
    """Chunks extracted text into smaller segments."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    return splitter.split_text(raw_text)
tools.append(chunk_data)

Embed and storing results in FAISS vector db tool

In [45]:
vector_db = None

@tool("embed_and_store")
def embed_and_store(chunks: list):
    """Embeds chunks and stores them in a FAISS vector database."""
    global vector_db
    vector_db = FAISS.from_texts(chunks, embeddings)
    return "Data successfully indexed in FAISS."
tools.append(embed_and_store)

Vecot search tool

In [46]:
@tool("vector_search")
def vector_search(query: str):
    """Retrieves top 5 results from FAISS for a given query."""
    if vector_db is None:
        return "Vector database is empty. Please index a PDF first."
    docs = vector_db.similarity_search(query, k=5)
    return "\n\n".join([d.page_content for d in docs])
tools.append(vector_search)

Youtube search

In [47]:
@tool("fetch_youtube_content", description="finds youtube videos using YoutubeSearchTool and then fetches content of the link using YoutubeLoader")
def fetch_youtube_content(query: str) -> str:
    """
    Fetches the full transcript text from a YouTube video link.
    """
    youtube_tool = YouTubeSearchTool()

    raw_result = youtube_tool.run(query)
    video_ids = ast.literal_eval(raw_result)
    links = [vid for vid in video_ids]

    all_text = ""
    i = 0
    for link in links:
      all_text += f"Video {i + 1} Content: "
      loader = YoutubeLoader.from_youtube_url(
          link,
          add_video_info=False,
      )
      doc = loader.load()
      all_text += doc[0].page_content + "\n"
      i = i + 1

    return all_text
tools.append(fetch_youtube_content)

Wikipedia search tool

In [48]:
wikiSearch = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
tools.append(wikiSearch)

google web search tool

In [49]:
# 1) Google Web Search
google_web = SearchApiAPIWrapper(
    searchapi_api_key=os.getenv("SEARCHAPI_API_KEY"),
    engine="google",            # standard Google web search
)

google_web_tool = Tool(
    name="google_web_search",
    description="General Google web search via SearchApi.io.",
    func=google_web.run,
)

tools.append(google_web_tool)

google news search tool

In [50]:
# 2) Google News
google_news = SearchApiAPIWrapper(
    searchapi_api_key=os.getenv("SEARCHAPI_API_KEY"),
    engine="google_news",       # Google News vertical
)

google_news_tool = Tool(
    name="google_news_search",
    description="Latest news headlines and articles via Google News (SearchApi.io).",
    func=google_news.run,
)

tools.append(google_news_tool)

Google scholar search

In [ ]:
# 3) Google Scholar
google_scholar = SearchApiAPIWrapper(
    searchapi_api_key=os.getenv("SEARCHAPI_API_KEY"),
    engine="google_scholar",    # Google Scholar vertical
)

google_scholar_tool = Tool(
    name="google_scholar_search",
    description="Academic papers and scholarly articles via Google Scholar (SearchApi.io).",
    func=google_scholar.run,
)

tools.append(google_scholar_tool)